# Etapa 5h — Mapas contra las lesiones

**Ronda de mejoras para la entrega final (desvío 14.11 del plan).** Solo desarrollo, la misma validación cruzada por personas. El conjunto de prueba no se usa.

**Motivo.** La retinopatía diabética es el grupo más débil (en prueba, −0,160 de Matthews contra sanos). Los falsos positivos están en lesiones (hemorragias, exudados) y en bordes con mucho contraste, como el del disco óptico. Ningún mapa actual distingue un vaso de una mancha o de un escalón de brillo: todos responden a "más claro que lo de al lado".

**Candidatas** (sobre el algoritmo con la normalización por división de `05g`; todo lo demás igual: mitad de resolución, entrenamiento equilibrado por grosor, valor fijo elegido en entrenamiento, 5 grupos de personas):
- **A.** Los 26 mapas actuales.
- **B.** A + **apertura por reconstrucción con segmentos** (Zana y Klein, 2001), con segmentos de 2 % y 4 % del diámetro del campo (2 mapas, 28 en total). Sobrevive lo que contiene un segmento largo en alguna dirección (los vasos) y recupera su forma con la reconstrucción; las manchas más chicas que el segmento desaparecen. Como el verde normalizado ya es un mapa, la regresión puede usar la diferencia entre los dos como indicador de "mancha".
- **C.** A + **borde asimétrico** (Zhang, Zhang, Zhang y Karray, 2010): |media local de la respuesta a la primera derivada de la gaussiana|, en las 5 escalas de la etapa 4 (5 mapas, 31 en total). Sobre un vaso, los dos bordes se cancelan; junto a un escalón (borde de una lesión o del disco), no.
- **D.** A + B + C (7 mapas, 33 en total).

**Medidas**, en las personas de afuera, a resolución original y con todos los píxeles del campo de la base: Matthews, Dice de línea central y sensibilidad de línea central en vasos finos. Descriptivo: Matthews y precisión (fracción de lo marcado que es vaso) por grupo, con atención a retinopatía.

**Regla de adopción (fijada antes de correr).** Una candidata reemplaza a A si
1. mejora Matthews con un intervalo al 95 % que no incluye el cero, **y**
2. no es inferior a A en vasos finos ni en Dice de línea central (límite inferior del intervalo de la diferencia > −0,03).

Si cumplen varias, gana la de menos mapas entre las no inferiores a la mejor en Matthews (margen 0,03). Si ninguna cumple, queda A.

**Control:** A tiene que reproducir el Matthews de B (división) en `05g` (0,7509).

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_value, fit_weights
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
APERTURA = C.LESION_FEATURES[:2]
BORDE = C.LESION_FEATURES[2:]
ALL = C.SEGMENT_FEATURES + APERTURA + BORDE
CANDS = {"A. 26 mapas": C.SEGMENT_FEATURES,
         "B. + apertura por reconstrucción": C.SEGMENT_FEATURES + APERTURA,
         "C. + borde asimétrico": C.SEGMENT_FEATURES + BORDE,
         "D. + las dos": ALL}
COLS = {c: [ALL.index(n) for n in f] for c, f in CANDS.items()}
ORDER = list(CANDS)
WORKERS = 4
print({c: len(f) for c, f in CANDS.items()}, "| normalización:", C.SEGMENT_BACKGROUND_MODE)

{'A. 26 mapas': 26, 'B. + apertura por reconstrucción': 28, 'C. + borde asimétrico': 31, 'D. + las dos': 33} | normalización: división


## Pasada 1: los 33 mapas en la muestra fija de cada imagen y los píxeles de entrenamiento de cada candidata

In [2]:
def pass1(image_id):
    from src.pipeline import sample_features, training_pixels
    v = sample_features(image_id, features=ALL)
    return image_id, {c: training_pixels(image_id, values=v[:, COLS[c]]) for c in CANDS}

t = time.time()
P1 = dict(map_images(pass1, ids, workers=WORKERS))
samples = {c: {i: P1[i][c] for i in ids} for c in CANDS}
del P1
print(f"Pasada 1: {time.time() - t:.0f} s")

Pasada 1: 220 s


## Ajuste por vuelta: pesos y valor fijo con las personas de entrenamiento

In [3]:
fit = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    for c in CANDS:
        w = fit_weights(samples[c], tr)
        full = np.zeros(1 + len(ALL)); full[0] = w[0]; full[1 + np.array(COLS[c])] = w[1:]   # pesos en los 33 mapas
        fit[(k, c)] = (full, choose_value(samples[c], w, tr))
print("Valor fijo elegido en cada vuelta:")
pd.DataFrame({c: [fit[(k, c)][1] for k in range(5)] for c in CANDS}, index=[f"vuelta {k}" for k in range(5)]).round(4)

Valor fijo elegido en cada vuelta:


,A. 26 mapas,B. + apertura por reconstrucción,C. + borde asimétrico,D. + las dos
vuelta 0,0.8768,0.8784,0.8772,0.8621
vuelta 1,0.8739,0.8751,0.8743,0.8753
vuelta 2,0.8761,0.8609,0.8764,0.8608
vuelta 3,0.8616,0.8626,0.8617,0.8627
vuelta 4,0.8697,0.8712,0.8700,0.8864


## Pasada 2: máscaras de las personas de afuera, a resolución original

In [4]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import feature_scores
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    rgb = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    auto = detect_fov_robust(rgb)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    probs, _, _ = feature_scores(rgb, [fit[(k, c)][0] for c in CANDS], 2, features=ALL)
    rows = []
    for c, p in zip(CANDS, probs):
        m = (upscale(p, fov.shape) > fit[(k, c)][1]) & auto
        marked = m & fov
        rows.append({"image": image_id, "candidata": c, "matthews": matthews(m, truth, fov),
                     "dice_linea_central": centerline_dice(m, truth, fov), "finos": m[thin].mean(),
                     "precision": (marked & truth).sum() / max(marked.sum(), 1)})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05h_lesiones.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

Pasada 2: 208 s


## Resultados

In [5]:
def compare(measures, base="A. 26 mapas"):
    out = []
    for meas in measures:
        w = R.pivot(index="image", columns="candidata", values=meas)[ORDER]
        d = w.sub(w[base], axis=0)
        lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
        for i, c in enumerate(ORDER):
            out.append({"medida": meas, "candidata": c, "media": w[c].mean(), "vs A": d[c].mean(),
                        "inferior": lo[i], "superior": hi[i]})
    return pd.DataFrame(out).set_index(["medida", "candidata"])

table = compare(["matthews", "dice_linea_central", "finos", "precision"])
table.round(4)

media    vs A  inferior  superior
medida             candidata                                                           
matthews           A. 26 mapas                       0.7509  0.0000    0.0000    0.0000
                   B. + apertura por reconstrucción  0.7520  0.0011    0.0005    0.0018
                   C. + borde asimétrico             0.7510  0.0001   -0.0000    0.0002
                   D. + las dos                      0.7515  0.0006   -0.0007    0.0019
dice_linea_central A. 26 mapas                       0.7557  0.0000    0.0000    0.0000
                   B. + apertura por reconstrucción  0.7563  0.0007   -0.0001    0.0016
                   C. + borde asimétrico             0.7557 -0.0000   -0.0001    0.0001
                   D. + las dos                      0.7558  0.0001   -0.0021    0.0021
finos              A. 26 mapas                       0.3522  0.0000    0.0000    0.0000
                   B. + apertura por reconstrucción  0.3514 -0.0008   -0.0051    0.0046
                   C. + borde asimétrico             0.3519 -0.0003   -0.0007    0.0001
                   D. + las dos                      0.3524  0.0002   -0.0084    0.0087
precision          A. 26 mapas                       0.7998  0.0000    0.0000    0.0000
                   B. + apertura por reconstrucción  0.7987 -0.0011   -0.0038    0.0010
                   C. + borde asimétrico             0.8000  0.0002    0.0000    0.0004
                   D. + las dos                      0.7979 -0.0019   -0.0057    0.0019

In [6]:
ok = {}
for c in ORDER[1:]:
    ok[c] = bool(table.loc[("matthews", c), "inferior"] > 0
                 and table.loc[("finos", c), "inferior"] > -C.NONINF_MARGIN_MATTHEWS
                 and table.loc[("dice_linea_central", c), "inferior"] > -C.NONINF_MARGIN_MATTHEWS)
print("Cumple la regla:", ok)
passing = [c for c in ORDER[1:] if ok[c]]
chosen = "A. 26 mapas"
if passing:
    w = R.pivot(index="image", columns="candidata", values="matthews")
    best = max(passing, key=lambda c: w[c].mean())
    d = w[passing].sub(w[best], axis=0)
    lo, _ = percentile_interval(person_bootstrap_means(d, dev["person"]))
    noninf = [c for c, l in zip(passing, lo) if l > -C.NONINF_MARGIN_MATTHEWS]
    chosen = min(noninf, key=lambda c: len(CANDS[c]))
print("Se adopta:", chosen)

Cumple la regla: {'B. + apertura por reconstrucción': True, 'C. + borde asimétrico': False, 'D. + las dos': False}
Se adopta: B. + apertura por reconstrucción


In [7]:
print("Por grupo (media):")
G = R.join(dev["group"], on="image")
G.pivot_table(index="candidata", columns="group", values=["matthews", "precision", "finos"]).loc[ORDER].round(3)

Por grupo (media):


finos               matthews               precision              
group                                dr      g      h       dr      g      h        dr      g      h
candidata                                                                                           
A. 26 mapas                       0.369  0.339  0.348    0.702  0.755  0.796     0.707  0.773  0.919
B. + apertura por reconstrucción  0.370  0.339  0.345    0.704  0.755  0.797     0.708  0.771  0.918
C. + borde asimétrico             0.369  0.339  0.348    0.702  0.755  0.796     0.708  0.773  0.919
D. + las dos                      0.372  0.340  0.346    0.703  0.754  0.797     0.707  0.770  0.917

## Decisión (escrita después de correr)

**Control:** A reproduce exacto el Matthews de la división en `05g` (0,7509).

**Se adopta B, la apertura por reconstrucción con segmentos** (28 mapas; `config.SEGMENT_FEATURES`), porque cumple la regla fijada. **El efecto es mínimo:**

| B contra A (personas de afuera) | Diferencia (intervalo 95 %) |
|---|---|
| Matthews | +0,0011 (+0,0005 a +0,0018) |
| Dice de línea central | +0,0007 (−0,0001 a +0,0016) |
| Vasos finos | −0,1 puntos (−0,5 a +0,5) |
| Precisión | −0,1 puntos (−0,4 a +0,1) |

Mejora Matthews en 24 de las 30 imágenes y en 15 de las 18 personas. En retinopatía, Matthews pasa de 0,702 a 0,704 y la precisión de 0,707 a 0,708: prácticamente igual. Se adopta por la regla (sin desvío de criterio); cuesta unos 6 segundos más por imagen.

**C, el borde asimétrico, no cambia nada** (+0,0001) y D no suma sobre B.

**Por qué los mapas contra las lesiones casi no ayudan en retinopatía** (análisis descriptivo con el modelo congelado en 8 imágenes de desarrollo): en las de retinopatía, entre el 51 % y el 79 % de los falsos positivos están a 3 píxeles o menos de un vaso de los expertos, y solo entre el 5 % y el 34 % lejos (más de 10 píxeles), que es donde estarían las lesiones sueltas. La mayor parte del "error" de retinopatía es de **ancho**: los expertos marcaron más angostos los vasos de ese grupo (etapa 1c, cociente marcado / visible 0,52 contra 0,80 en sanos), y el algoritmo marca el vaso visible. Un mapa que borra manchas no cambia eso. Coincide con el desvío 14.3: parte de la diferencia por grupo es la forma de marcar.

**Cuidados.** Elección hecha con las mismas 30 imágenes; el conjunto de prueba no se usó.

**Qué cambia en el código.** `SEGMENT_FEATURES` pasa a 28 mapas. El archivo de pesos guarda su lista de mapas y `segment_final` usa esa (`load_feature_names`): los pesos congelados (26 mapas, resta) dan exactamente la misma probabilidad que antes.